# Busca semantica de cartas (Etapa 3.2)

*(Feito com ClaudeAI: https://claude.ai/share/b3536ad5-de2f-4b3f-a788-ba1c9a17f7da)*

Usa os embeddings gerados na etapa 3.1 pra montar uma funcao de busca de
verdade: consulta em linguagem natural -> vetor -> similaridade de
cosseno -> top-k cartas, com nome/tipo/custo/texto formatados pra leitura,
nao so uma lista crua de scores.

Basta ajustar os caminhos na celula de configuracao e rodar tudo
(`Run All`). A ultima celula e' interativa -- digite sua propria consulta.

## Configuracao

In [2]:
import json
import numpy as np

CARD_DOCUMENTS_PATH = "Dados/Etapa1_card_expanded/card_documents.jsonl"
EMBEDDINGS_PATH = "Dados/Etapa2_embeddings/embeddings.npy"
MODEL_NAME = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

## Carregar embeddings e dados completos da carta

Os embeddings (etapa 3.1) foram gerados na mesma ordem em que
`card_documents.jsonl` lista as cartas -- filtrando so as que tem
`embedding_text_structured` (mesmo criterio usado pra gerar os vetores).
Reconstruimos essa lista de nomes direto do arquivo, sem depender de um
segundo arquivo de metadados separado (mais facil de esquecer de enviar
junto e desnecessario, ja que a ordem e' sempre a mesma).

In [3]:
embeddings = np.load(EMBEDDINGS_PATH)

names = []
cards_by_name = {}
with open(CARD_DOCUMENTS_PATH, encoding="utf-8") as f:
    for line in f:
        card = json.loads(line)
        if not card.get("embedding_text_structured"):
            continue
        names.append(card["name"])
        cards_by_name.setdefault(card["name"], card)

assert len(names) == embeddings.shape[0], (
    f"Descompasso: {len(names)} nomes vs {embeddings.shape[0]} vetores -- "
    "os embeddings provavelmente foram gerados a partir de uma versao "
    "diferente do card_documents.jsonl. Regere os embeddings (etapa 3.1)."
)
print(f"{len(names)} vetores carregados, shape {embeddings.shape}")

33147 vetores carregados, shape (33147, 384)


## Carregar o modelo

Mesmo modelo da etapa 3.1 -- precisa ser o mesmo, porque vetores gerados
por modelos diferentes nao sao comparaveis entre si.

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)
print(f"Modelo carregado: {MODEL_NAME}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo carregado: BAAI/bge-small-en-v1.5


## Funcao de busca

`search(query, k)` embeda a consulta (com o prefixo que os modelos BGE
esperam pra busca -- documentado na etapa 3.1), calcula similaridade de
cosseno contra os 33 mil vetores de uma vez (produto de matriz, rapido o
suficiente sem precisar de indice tipo FAISS nesse tamanho de dataset) e
devolve os top-k com os dados completos da carta.

In [10]:
def search(query: str, k: int = 10):
    query_vec = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    scores = embeddings @ query_vec
    top_k = np.argsort(-scores)[:k]

    results = []
    for i in top_k:
        name = names[i]
        card = cards_by_name.get(name)
        if card is None:
            continue
        results.append({
            "score": float(scores[i]),
            "name": name,
            "type_line": card.get("type_line"),
            "mana_cost": card.get("mana_cost"),
            "oracle_text": card.get("oracle_text"),
        })
    return results


def print_results(results):
    for r in results:
        print(f"{r['score']:.3f}  {r['name']}  ({r['mana_cost'] or '-'})")
        print(f"       {r['type_line']}")
        if r["oracle_text"]:
            print(f"       {r['oracle_text'][:120]}")
        print()

## Testando com algumas consultas

In [11]:
for query in [
    "creature that destroys artifacts",
    "cheap green mana creature",
    "counter target spell",
    "deals damage to any target",
]:
    print(f"=== {query!r} ===")
    print_results(search(query, k=5))

=== 'creature that destroys artifacts' ===
0.777  Darksteel Brute  ({2})
       Artifact
       Indestructible (Damage and effects that say "destroy" don't destroy this artifact.)
{3}: This artifact becomes a 2/2 Bea

0.775  Purge  ({1}{W})
       Instant
       Destroy target artifact creature or black creature. It can't be regenerated.

0.772  Darksteel Myr  ({3})
       Artifact Creature — Myr
       Indestructible (Damage and effects that say "destroy" don't destroy this creature. If its toughness is 0 or less, it sti

0.769  Predator Ooze  ({G}{G}{G})
       Creature — Ooze
       Indestructible (Damage and effects that say "destroy" don't destroy this creature.)
Whenever this creature attacks, put 

0.766  Oxidda Scrapmelter  ({3}{R})
       Creature — Beast
       When this creature enters, destroy target artifact.

=== 'cheap green mana creature' ===
0.827  Redwood Treefolk  ({4}{G})
       Creature — Treefolk

0.823  Tendril of the Mycotyrant  ({1}{G})
       Creature — Fungus

## Sua consulta

Troque o texto abaixo e rode a celula quantas vezes quiser.

In [22]:
print_results(search("gain control of permanent and ", k=10))

0.706  Exert Influence  ({4}{U})
       Sorcery
       Converge — Gain control of target creature if its power is less than or equal to the number of colors of mana spent to c

0.692  Dominate  ({X}{1}{U}{U})
       Instant
       Gain control of target creature with mana value X or less.

0.692  Unexpected Request  ({2}{R})
       Sorcery
       Gain control of target creature until end of turn. Untap that creature. It gains haste until end of turn. You may attach

0.686  Incessant Provocation  ({3}{R})
       Sorcery
       Gain control of target creature until end of turn. Untap that creature. It gains haste until end of turn. It perpetually

0.685  Mass Manipulation  ({X}{X}{U}{U}{U}{U})
       Sorcery
       Gain control of X target creatures and/or planeswalkers.

0.680  Wrangle  ({1}{R})
       Sorcery
       Gain control of target creature with power 4 or less until end of turn. Untap that creature. It gains haste until end of

0.679  Harmless Offering  ({2}{R})
       Sorcery
